# Feature Review and Preprocessing Assessment

This notebook reviews the dataset features before model development. The analysis focuses on feature types, missing and unknown values, identifier columns, high-missingness features, potential leakage concerns, and preprocessing requirements.

No final preprocessing or feature-selection decisions are made in this notebook. These decisions will be discussed and finalized by the team.

In [1]:
import pandas as pd

DATA_PATH = "../data/raw/diabetic_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

Dataset shape: (101766, 50)
Number of columns: 50


## 1. Feature Inventory

The dataset contains identifiers, demographic variables, admission information, healthcare utilization variables, clinical activity variables, diagnosis-related variables, diabetes testing variables, medication variables, and treatment-change variables.


In [2]:
print("===== ALL FEATURES =====")

for i, col in enumerate(df.columns, start=1):
    print(f"{i}. {col}")

===== ALL FEATURES =====
1. encounter_id
2. patient_nbr
3. race
4. gender
5. age
6. weight
7. admission_type_id
8. discharge_disposition_id
9. admission_source_id
10. time_in_hospital
11. payer_code
12. medical_specialty
13. num_lab_procedures
14. num_procedures
15. num_medications
16. number_outpatient
17. number_emergency
18. number_inpatient
19. diag_1
20. diag_2
21. diag_3
22. number_diagnoses
23. max_glu_serum
24. A1Cresult
25. metformin
26. repaglinide
27. nateglinide
28. chlorpropamide
29. glimepiride
30. acetohexamide
31. glipizide
32. glyburide
33. tolbutamide
34. pioglitazone
35. rosiglitazone
36. acarbose
37. miglitol
38. troglitazone
39. tolazamide
40. examide
41. citoglipton
42. insulin
43. glyburide-metformin
44. glipizide-metformin
45. glimepiride-pioglitazone
46. metformin-rosiglitazone
47. metformin-pioglitazone
48. change
49. diabetesMed
50. readmitted


## 2. Target Variable

The modeling target is `readmitted_30`, where:

- `0` = patient was not readmitted within 30 days
- `1` = patient was readmitted within 30 days

The original `readmitted` column is used to create this target.

In [3]:
df["readmitted_30"] = (df["readmitted"] == "<30").astype(int)

print(df["readmitted_30"].value_counts())

readmitted_30
0    90409
1    11357
Name: count, dtype: int64


## 3. Data Type Review

Data types are reviewed to identify numerical and categorical features before preprocessing.

In [4]:
print("===== DATA TYPES =====")

dtype_summary = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str)
})

print(dtype_summary.to_string(index=False))

===== DATA TYPES =====
                 feature dtype
            encounter_id int64
             patient_nbr int64
                    race   str
                  gender   str
                     age   str
                  weight   str
       admission_type_id int64
discharge_disposition_id int64
     admission_source_id int64
        time_in_hospital int64
              payer_code   str
       medical_specialty   str
      num_lab_procedures int64
          num_procedures int64
         num_medications int64
       number_outpatient int64
        number_emergency int64
        number_inpatient int64
                  diag_1   str
                  diag_2   str
                  diag_3   str
        number_diagnoses int64
           max_glu_serum   str
               A1Cresult   str
               metformin   str
             repaglinide   str
             nateglinide   str
          chlorpropamide   str
             glimepiride   str
           acetohexamide   str
               g

In [7]:
numeric_features = df.select_dtypes(include=["number"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object"]).columns.tolist()

# Remove the target from the feature lists
numeric_features = [
    col for col in numeric_features
    if col != "readmitted_30"
]

print("Number of numerical feature columns:", len(numeric_features))
print("Number of categorical feature columns:", len(categorical_features))

print("\nNumerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Number of numerical feature columns: 13
Number of categorical feature columns: 37

Numerical features:
['encounter_id', 'patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Categorical features:
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


/var/folders/67/m53prws178q2hzd87j36jqgh0000gn/T/ipykernel_3001/1107067299.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df.select_dtypes(include=["object"]).columns.tolist()


## 4. Unknown and Missing Value Review

The dataset uses both standard missing values and the value `?` to represent unknown or unavailable information. These values need to be considered during preprocessing.

In [8]:
question_mark_counts = (df == "?").sum()

question_mark_counts = (
    question_mark_counts[question_mark_counts > 0]
    .sort_values(ascending=False)
)

print("===== FEATURES CONTAINING '?' =====")
print(question_mark_counts)

===== FEATURES CONTAINING '?' =====
weight               98569
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64


In [9]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

print("===== STANDARD MISSING VALUES =====")
print(missing_summary)

===== STANDARD MISSING VALUES =====
               missing_count  missing_percentage
max_glu_serum          96420               94.75
A1Cresult              84748               83.28


In [10]:
unknown_summary = pd.DataFrame({
    "question_mark_count": (df == "?").sum(),
    "question_mark_percentage": ((df == "?").mean() * 100).round(2),
    "nan_count": df.isna().sum(),
    "nan_percentage": (df.isna().mean() * 100).round(2)
})

unknown_summary = unknown_summary[
    (unknown_summary["question_mark_count"] > 0) |
    (unknown_summary["nan_count"] > 0)
].sort_values(
    "question_mark_percentage",
    ascending=False
)

print("===== MISSING / UNKNOWN SUMMARY =====")
print(unknown_summary)

===== MISSING / UNKNOWN SUMMARY =====
                   question_mark_count  question_mark_percentage  nan_count  \
weight                           98569                     96.86          0   
medical_specialty                49949                     49.08          0   
payer_code                       40256                     39.56          0   
race                              2273                      2.23          0   
diag_3                            1423                      1.40          0   
diag_2                             358                      0.35          0   
diag_1                              21                      0.02          0   
max_glu_serum                        0                      0.00      96420   
A1Cresult                            0                      0.00      84748   

                   nan_percentage  
weight                       0.00  
medical_specialty            0.00  
payer_code                   0.00  
race                       

## 5. Identifier Features

Identifier variables are reviewed separately because they identify patients or encounters rather than directly representing clinical characteristics.

These variables require special consideration before modeling.

In [11]:
id_features = [
    "encounter_id",
    "patient_nbr"
]

print("===== IDENTIFIER FEATURES =====")

for col in id_features:
    print(f"\n{col}")
    print("Unique values:", df[col].nunique())
    print("Total rows:", len(df))

===== IDENTIFIER FEATURES =====

encounter_id
Unique values: 101766
Total rows: 101766

patient_nbr
Unique values: 71518
Total rows: 101766


## 6. High-Missingness Features

Features with a large proportion of unknown or missing values are reviewed for their potential impact on preprocessing and modeling.

In [12]:
high_missing_features = unknown_summary[
    (unknown_summary["question_mark_percentage"] >= 20) |
    (unknown_summary["nan_percentage"] >= 20)
]

print("===== HIGH-MISSINGNESS FEATURES =====")
print(high_missing_features)

===== HIGH-MISSINGNESS FEATURES =====
                   question_mark_count  question_mark_percentage  nan_count  \
weight                           98569                     96.86          0   
medical_specialty                49949                     49.08          0   
payer_code                       40256                     39.56          0   
max_glu_serum                        0                      0.00      96420   
A1Cresult                            0                      0.00      84748   

                   nan_percentage  
weight                       0.00  
medical_specialty            0.00  
payer_code                   0.00  
max_glu_serum               94.75  
A1Cresult                   83.28  


## 7. Potential Leakage and Prediction Timing Review

Some variables may contain information that becomes available only later in the hospital encounter.

For example, `discharge_disposition_id` should be reviewed carefully because discharge-related information may not be available at the intended prediction time.

This notebook does not make a final decision to remove the feature. The prediction timing and feature availability should be discussed by the team.

In [13]:
timing_sensitive_features = [
    "admission_type_id",
    "admission_source_id",
    "discharge_disposition_id"
]

print("===== TIMING-SENSITIVE FEATURES =====")

for col in timing_sensitive_features:
    if col in df.columns:
        print(f"{col}: present")

===== TIMING-SENSITIVE FEATURES =====
admission_type_id: present
admission_source_id: present
discharge_disposition_id: present


## 8. Preprocessing Assessment

| Feature / Group | Observation | Preprocessing Consideration | Final Decision |
|---|---|---|---|
| `encounter_id` | Identifier | Review whether it should be excluded from modeling | Team decision |
| `patient_nbr` | Patient identifier | Review whether it should be excluded from modeling | Team decision |
| `race` | Contains unknown values | Missing/unknown handling required | Team decision |
| `weight` | Very high proportion of unknown values | High-missingness handling required | Team decision |
| `payer_code` | High proportion of unknown values | Missing/unknown handling required | Team decision |
| `medical_specialty` | High proportion of unknown values | Missing/unknown handling required | Team decision |
| Categorical features | Multiple categorical variables | Encoding required before many models | Team decision |
| Numerical features | Different distributions and scales | Distribution/scaling review may be required | Team decision |
| `discharge_disposition_id` | Discharge-related information | Review for prediction-time availability | Team decision |

## 9. Feature Review Findings

### Main Observations

- The dataset contains both numerical and categorical features.
- Several variables contain `?` values representing unknown information.
- Some variables contain standard `NaN` values.
- `weight`, `medical_specialty`, and `payer_code` require particular attention because of their high proportion of unknown values.
- `encounter_id` and `patient_nbr` are identifiers and require separate consideration before modeling.
- Categorical features will require an appropriate encoding strategy.
- Numerical features have different distributions and may require additional preprocessing depending on the selected model.
- `discharge_disposition_id` should be reviewed carefully because prediction-time availability may create a potential leakage concern.

### Important Note

This review identifies preprocessing requirements but does not make final preprocessing or feature-selection decisions.

The final decisions regarding missing-value handling, encoding, scaling, feature removal, and feature selection will be made jointly by the team before model development.

In [14]:
feature_review_summary = pd.DataFrame({
    "Category": [
        "Numerical features",
        "Categorical features",
        "High unknown/missingness",
        "Identifier features",
        "Potential timing concern"
    ],
    "Observation": [
        f"{len(numeric_features)} numerical feature columns identified",
        f"{len(categorical_features)} categorical feature columns identified",
        "weight, medical_specialty, payer_code, max_glu_serum, A1Cresult",
        "encounter_id and patient_nbr",
        "discharge_disposition_id requires prediction-time review"
    ]
})

print("===== FEATURE REVIEW SUMMARY =====")
print(feature_review_summary.to_string(index=False))

===== FEATURE REVIEW SUMMARY =====
                Category                                                     Observation
      Numerical features                         13 numerical feature columns identified
    Categorical features                       37 categorical feature columns identified
High unknown/missingness weight, medical_specialty, payer_code, max_glu_serum, A1Cresult
     Identifier features                                    encounter_id and patient_nbr
Potential timing concern        discharge_disposition_id requires prediction-time review


## 10. Conclusion

The feature review identified the main data-quality and preprocessing considerations before model development.

The dataset contains a mixture of numerical and categorical features, along with identifiers, unknown values represented by `?`, and standard missing values represented by `NaN`.

The main areas requiring team discussion are:

- Handling high-missingness features
- Handling `?` and `NaN` values
- Encoding categorical features
- Treatment of identifier columns
- Reviewing skewed numerical features
- Checking feature availability at prediction time
- Final feature selection

No preprocessing or feature-selection decision is finalized in this notebook. The observations will be used as input for the team's preprocessing and modeling stage.